# Ambulance Green Corridor — GRPO Training
**OpenEnv Hackathon 2026**

Trains `Qwen2.5-0.5B-Instruct` via GRPO to:
1. Choose the correct specialist hospital for the patient
2. Clear traffic signals efficiently (only toggle wrong-phase ones)

**Runtime:** T4 GPU (free Colab tier)  
**Expected time:** ~35–45 min

Expected improvement after ~60 iterations:
| Metric | Before | After |
|---|---|---|
| Reward | ~900 | ~1600 |
| Arrival rate | ~60% | ~95% |
| Signal efficiency | ~20% | ~85% |

In [ ]:
# CELL 1 — Install dependencies
# Runtime will restart after this cell — that's expected, continue from Cell 2
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate
!pip install -q "openenv-core[core]>=0.2.2"
!pip install -q nest_asyncio

# Clone the final branch (ambulance_env lives there)
!git clone -q -b final https://github.com/ajitg25/openEnv-hackathon.git /content/openEnv-hackathon

import sys
sys.path.insert(0, '/content/openEnv-hackathon/envs')
print('Install complete. Runtime will restart — re-run from Cell 2.')

In [ ]:
# CELL 2 — Imports & server startup
import sys, os
from pathlib import Path

# Re-add path after runtime restart (Colab clears sys.path on restart)
REPO_ROOT = Path('/content/openEnv-hackathon')
ENVS_PATH = str(REPO_ROOT / 'envs')
if ENVS_PATH not in sys.path:
    sys.path.insert(0, ENVS_PATH)

# Verify the clone exists — if not, re-run Cell 1
if not REPO_ROOT.exists():
    raise RuntimeError('Repo not found — re-run Cell 1 first, then this cell')
print('Repo found:', REPO_ROOT)
print('envs:', [p.name for p in (REPO_ROOT / 'envs').iterdir() if p.is_dir()])

import json, re, subprocess, time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.optim import AdamW

from ambulance_env import AmbulanceEnv
from ambulance_env.models import AmbulanceAction, SignalControl

ENV_URL = 'http://localhost:8000'
DIFFICULTY = 'easy'  # easy | medium | hard

print('Starting ambulance_env server...')
_server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'ambulance_env.server.app:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'error'],
    env={**os.environ, 'PYTHONPATH': ENVS_PATH, 'AMBULANCE_DIFFICULTY': DIFFICULTY},
)
time.sleep(4)
print('Server ready at', ENV_URL)

In [ ]:
# CELL 3 — Load model with Unsloth
from unsloth import FastLanguageModel

MODEL_NAME = 'unsloth/Qwen2.5-0.5B-Instruct'  # ~1.5 GB VRAM, fits free T4
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Model: {MODEL_NAME}')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# CELL 4 — Prompt formatters
SYSTEM_PROMPT = (
    "You are an emergency services AI managing an ambulance in a real city.\n"
    "You must:\n"
    "  1. Choose the best hospital (consider specialization, distance, current traffic, road quality)\n"
    "  2. Clear traffic signals ahead — but ONLY signals in the WRONG phase\n"
    "  3. Re-route dynamically if traffic spikes, accidents, or road closures appear\n"
    "  4. Switch hospitals mid-journey if an alternative route becomes significantly faster\n\n"
    "Heavy traffic slows the ambulance even on green signals. Potholed roads force slow speeds.\n"
    "Be precise. Follow the output format exactly."
)

def _road_quality_label(q: float) -> str:
    if q >= 0.9: return "highway"
    if q >= 0.65: return "good"
    if q >= 0.4: return "moderate"
    return "POTHOLED"

def format_prompt(obs) -> str:
    lines = []

    # --- Patient ---
    lines.append(f"=== EMERGENCY DISPATCH ===")
    lines.append(f"Patient  : {obs.patient_location} | condition: {obs.patient_condition}")
    lines.append(f"Ambulance: {obs.ambulance_location} | time: {obs.time_elapsed_seconds:.0f}s / {obs.time_limit_seconds:.0f}s")
    lines.append("")

    # --- Active events ---
    if obs.active_events:
        lines.append("⚠ DYNAMIC EVENTS:")
        for e in obs.active_events:
            lines.append(f"  [{e.event_type.upper()}] at {e.position} — {e.description}")
        lines.append("")

    # --- Current route ---
    if obs.target_hospital_id:
        r = obs.current_route
        lines.append(f"CURRENT ROUTE → {obs.target_hospital_id}")
        lines.append(f"  ETA={r.estimated_time:.0f}s | segments={len(r.segments)} | "
                     f"damaged={r.num_damaged_segments} | heavy_traffic={r.num_heavy_traffic_segments}")
        if r.segments:
            lines.append("  Next segments:")
            for seg in r.segments[:4]:
                lines.append(f"    {seg.from_pos}→{seg.to_pos} | {seg.road_type} "
                              f"| quality={_road_quality_label(seg.road_quality)} "
                              f"| traffic={seg.traffic_volume:.0%} "
                              f"| est={seg.estimated_transit_time:.0f}s"
                              + (" [BLOCKED]" if seg.blocked else ""))
        lines.append("")

    # --- Alternative routes ---
    if obs.alternative_routes:
        lines.append("ALTERNATIVES (consider switching if ETA much lower):")
        for alt in obs.alternative_routes:
            hosp = next((h for h in obs.hospitals if h.hospital_id == alt.hospital_id), None)
            spec = hosp.specialization if hosp else "?"
            match = " ← specialist match" if hosp and hosp.specialization == obs.patient_condition else ""
            lines.append(f"  {alt.hospital_id} ({spec}){match}: ETA={alt.estimated_time:.0f}s | "
                         f"damaged={alt.num_damaged_segments} | heavy={alt.num_heavy_traffic_segments}")
        lines.append("")

    # --- Hospital options (always shown for reference) ---
    lines.append("HOSPITALS:")
    for h in obs.hospitals:
        cap = " [AT CAPACITY]" if h.at_capacity else ""
        match = " ← specialist match" if h.specialization == obs.patient_condition else ""
        lines.append(f"  {h.hospital_id}: {h.name} | spec={h.specialization} | "
                     f"dist={h.distance_to_patient} | est={h.travel_time_estimate:.0f}s{cap}{match}")
    lines.append("")

    # --- Traffic signals ---
    if obs.lookahead_signals:
        lines.append("NEXT SIGNALS (only change WRONG ones):")
        for s in obs.lookahead_signals:
            needed = "ns_green" if s.ambulance_direction in ("north", "south") else "ew_green"
            status = "OK" if s.phase == needed else f"WRONG — needs {needed}"
            lines.append(f"  ({s.row},{s.col}): current={s.phase} | dir={s.ambulance_direction} | {status} | density={s.traffic_density:.0%}")
        lines.append("")

    # --- Current segment ---
    if obs.current_segment:
        seg = obs.current_segment
        lines.append(f"CURRENT ROAD: {seg.road_type} | quality={_road_quality_label(seg.road_quality)} | traffic={seg.traffic_volume:.0%}")
        lines.append(f"  (speed factor ≈ {obs.last_speed_factor:.0%} of max)")
        lines.append("")

    # --- Performance ---
    lines.append(f"STATS: stops_at_red={obs.stops_at_red} | "
                 f"signal_efficiency={obs.signal_efficiency:.0%} | "
                 f"unnecessary_toggles={obs.unnecessary_toggles}")
    lines.append("")

    # --- Instructions ---
    if not obs.target_hospital_id:
        lines.append("ACTION: Choose a hospital. Reply as JSON:")
        lines.append('{"hospital_id": "hosp_X", "signal_controls": [], "preferred_direction": null}')
    else:
        lines.append("ACTION: Manage signals and optionally switch hospital or set preferred direction.")
        lines.append("  - Only include signals with status=WRONG in signal_controls")
        lines.append("  - Set hospital_id only if switching to a faster alternative")
        lines.append("  - Set preferred_direction (north/south/east/west) to force a turn")
        lines.append('Reply as JSON: {"hospital_id": null, "signal_controls": [{"row": R, "col": C, "phase": "ns_green_or_ew_green"}], "preferred_direction": null}')

    return "\n".join(lines)

def build_chat(obs) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": format_prompt(obs)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("Prompt formatters ready.")

In [ ]:
# CELL 5 — Action parser
def parse_action(response_text: str, obs) -> AmbulanceAction:
    text = response_text.strip()

    # Try JSON first (works for all phases)
    try:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            data = json.loads(m.group())

            # Hospital selection / switch
            hospital_id = data.get("hospital_id")
            if hospital_id:
                valid_ids = {h.hospital_id for h in obs.hospitals if not h.at_capacity}
                if hospital_id not in valid_ids:
                    hospital_id = None

            # Signal controls — only keep valid phases
            controls = [
                SignalControl(row=int(c["row"]), col=int(c["col"]), phase=c["phase"])
                for c in data.get("signal_controls", [])
                if c.get("phase") in ("ns_green", "ew_green")
            ]

            # Preferred direction
            direction = data.get("preferred_direction")
            if direction not in ("north", "south", "east", "west"):
                direction = None

            return AmbulanceAction(
                hospital_id=hospital_id,
                signal_controls=controls,
                preferred_direction=direction,
            )
    except (json.JSONDecodeError, KeyError, ValueError, TypeError):
        pass

    # Fallback: no hospital selected yet → pick best available
    if not obs.target_hospital_id:
        available = [h for h in obs.hospitals if not h.at_capacity]
        specialists = [h for h in available if h.specialization == obs.patient_condition]
        pool = specialists if specialists else available
        if pool:
            best = min(pool, key=lambda h: h.travel_time_estimate)
            return AmbulanceAction(hospital_id=best.hospital_id)

    # Fallback: routing — clear only wrong-phase signals
    controls = [
        SignalControl(
            row=s.row, col=s.col,
            phase="ns_green" if s.ambulance_direction in ("north", "south") else "ew_green",
        )
        for s in obs.lookahead_signals
        if s.phase != ("ns_green" if s.ambulance_direction in ("north", "south") else "ew_green")
    ]
    return AmbulanceAction(signal_controls=controls)

print("Action parser ready.")

In [ ]:
# CELL 6 — Episode rollout (async client)
import asyncio

@torch.no_grad()
async def collect_episode_async(temperature=0.8, max_new_tokens=256):
    env = AmbulanceEnv(base_url=ENV_URL)
    steps = []
    try:
        result = await env.reset()
        obs = result.observation
        while not result.done:
            prompt = build_chat(obs)
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            output_ids = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                temperature=temperature, do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
            new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
            response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
            action = parse_action(response_text, obs)
            result = await env.step(action)
            obs = result.observation
            steps.append({
                "prompt": prompt,
                "response": response_text,
                "step_reward": float(result.reward or 0.0),
            })
        total = sum(s["step_reward"] for s in steps)
        for s in steps:
            s["episode_reward"] = total
        state = env.state   # property, not method
        return steps, state
    finally:
        await env.close()

def collect_episode(temperature=0.8, max_new_tokens=256):
    """Sync wrapper — handles Colab's always-running event loop via nest_asyncio."""
    import nest_asyncio
    nest_asyncio.apply()
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(collect_episode_async(temperature, max_new_tokens))

print("collect_episode() ready.")

In [ ]:
# CELL 7 — Baseline evaluation (BEFORE training)
def evaluate(num_episodes=8):
    rewards, arrivals, efficiencies, times, reroutes = [], [], [], [], []
    for _ in range(num_episodes):
        steps, state = collect_episode(temperature=0.1)
        rewards.append(steps[-1]["episode_reward"] if steps else 0.0)
        arrivals.append(float(state.success))
        efficiencies.append(state.signal_efficiency)
        times.append(state.arrival_time or 999.0)
        reroutes.append(getattr(state, "successful_reroutes", 0))
    return {
        "mean_reward":     float(np.mean(rewards)),
        "arrival_rate":    float(np.mean(arrivals)),
        "mean_efficiency": float(np.mean(efficiencies)),
        "mean_time":       float(np.mean(times)),
        "mean_reroutes":   float(np.mean(reroutes)),
    }

print("Running baseline evaluation (8 episodes)...")
baseline = evaluate(num_episodes=8)
print(f"BASELINE  reward={baseline['mean_reward']:.1f}  "
      f"arrival={baseline['arrival_rate']:.0%}  "
      f"efficiency={baseline['mean_efficiency']:.0%}  "
      f"reroutes={baseline['mean_reroutes']:.1f}  "
      f"time={baseline['mean_time']:.0f}s")

In [ ]:
# CELL 8 — GRPO Training loop
NUM_ITERATIONS = 60
GROUP_SIZE     = 4
BETA_KL        = 0.01
LR             = 5e-5

optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
history = {
    "iteration": [], "mean_reward": [], "arrival_rate": [],
    "signal_efficiency": [], "mean_time": [], "mean_reroutes": [],
}

print(f"GRPO training: {NUM_ITERATIONS} iterations x {GROUP_SIZE} episodes\n")

for iteration in range(NUM_ITERATIONS):
    model.eval()
    group_steps, group_states = [], []
    for _ in range(GROUP_SIZE):
        steps, state = collect_episode(temperature=0.8)
        group_steps.append(steps)
        group_states.append(state)

    episode_rewards = [s[-1]["episode_reward"] if s else 0.0 for s in group_steps]
    r_tensor = torch.tensor(episode_rewards)
    advantages = (r_tensor - r_tensor.mean()) / (r_tensor.std() + 1e-8)

    model.train()
    iter_loss, num_updates = 0.0, 0

    for steps, adv in zip(group_steps, advantages.tolist()):
        for step in steps:
            prompt_ids   = tokenizer(step["prompt"],   return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN-256).input_ids.to(model.device)
            response_ids = tokenizer(step["response"], return_tensors="pt", truncation=True, max_length=256).input_ids.to(model.device)
            if response_ids.shape[1] == 0:
                continue
            full_ids = torch.cat([prompt_ids, response_ids], dim=1)
            with torch.cuda.amp.autocast(dtype=torch.bfloat16):
                logits = model(full_ids).logits
            resp_logits = logits[:, prompt_ids.shape[1]-1:-1, :]
            log_probs   = F.log_softmax(resp_logits, dim=-1)
            token_lp    = log_probs.gather(2, response_ids.unsqueeze(-1)).squeeze(-1)
            mean_lp     = token_lp.mean()
            loss = -adv * mean_lp + BETA_KL * (mean_lp ** 2)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            iter_loss += loss.item()
            num_updates += 1

    mean_reward  = float(np.mean(episode_rewards))
    arrival_rate = float(np.mean([s.success for s in group_states]))
    mean_eff     = float(np.mean([s.signal_efficiency for s in group_states]))
    mean_time    = float(np.mean([s.arrival_time or 999.0 for s in group_states]))
    mean_reroutes = float(np.mean([getattr(s, "successful_reroutes", 0) for s in group_states]))

    history["iteration"].append(iteration + 1)
    history["mean_reward"].append(mean_reward)
    history["arrival_rate"].append(arrival_rate)
    history["signal_efficiency"].append(mean_eff)
    history["mean_time"].append(mean_time)
    history["mean_reroutes"].append(mean_reroutes)

    print(f"[{iteration+1:3d}/{NUM_ITERATIONS}]  reward={mean_reward:7.1f}  "
          f"arrival={arrival_rate:.0%}  efficiency={mean_eff:.0%}  "
          f"reroutes={mean_reroutes:.1f}  time={mean_time:5.0f}s  "
          f"loss={iter_loss/max(1,num_updates):.4f}")

In [ ]:
# CELL 9 — Final evaluation (AFTER training)
print("Running final evaluation...")
final = evaluate(num_episodes=8)
print(f"FINAL     reward={final['mean_reward']:.1f}  arrival={final['arrival_rate']:.0%}  "
      f"efficiency={final['mean_efficiency']:.0%}  reroutes={final['mean_reroutes']:.1f}  time={final['mean_time']:.0f}s")

print("\n── Improvement Summary ─────────────────────────────────")
print(f"  Reward        : {baseline['mean_reward']:6.1f}  →  {final['mean_reward']:6.1f}  ({final['mean_reward']-baseline['mean_reward']:+.1f})")
print(f"  Arrival rate  : {baseline['arrival_rate']:.0%}       →  {final['arrival_rate']:.0%}")
print(f"  Efficiency    : {baseline['mean_efficiency']:.0%}       →  {final['mean_efficiency']:.0%}")
print(f"  Reroutes/ep   : {baseline['mean_reroutes']:.1f}         →  {final['mean_reroutes']:.1f}  (agent learns when to switch)")
print(f"  Travel time   : {baseline['mean_time']:.0f}s       →  {final['mean_time']:.0f}s  ({final['mean_time']-baseline['mean_time']:+.0f}s)")

In [ ]:
# CELL 10 — Training plots (4 panels)
def smooth(values, window=5):
    if len(values) < window:
        return np.array(values)
    return np.convolve(values, np.ones(window)/window, mode="valid")

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle("Ambulance Green Corridor — GRPO Training", fontsize=14, fontweight="bold")
iters  = history["iteration"]
sm_off = 4

# Plot 1: Reward
ax = axes[0]
ax.plot(iters, history["mean_reward"], alpha=0.25, color="royalblue")
ax.plot(iters[sm_off:], smooth(history["mean_reward"]), color="royalblue", linewidth=2, label="Trained")
ax.axhline(baseline["mean_reward"], color="red",   linestyle="--", linewidth=1.5, label=f"Baseline ({baseline['mean_reward']:.0f})")
ax.axhline(final["mean_reward"],    color="green", linestyle="--", linewidth=1.5, label=f"Final ({final['mean_reward']:.0f})")
ax.set_xlabel("Training Episode"); ax.set_ylabel("Episode Reward")
ax.set_title("Episode Reward"); ax.legend(fontsize=8)

# Plot 2: Arrival rate
ax = axes[1]
ax.plot(iters, [v*100 for v in history["arrival_rate"]], alpha=0.25, color="darkorange")
ax.plot(iters[sm_off:], smooth([v*100 for v in history["arrival_rate"]]), color="darkorange", linewidth=2)
ax.axhline(baseline["arrival_rate"]*100, color="red",   linestyle="--", linewidth=1.5, label=f"Before ({baseline['arrival_rate']:.0%})")
ax.axhline(final["arrival_rate"]*100,    color="green", linestyle="--", linewidth=1.5, label=f"After ({final['arrival_rate']:.0%})")
ax.set_xlabel("Training Episode"); ax.set_ylabel("Arrival Rate (%)")
ax.set_title("Hospital Arrival Rate"); ax.set_ylim(0, 105); ax.legend(fontsize=8)

# Plot 3: Signal efficiency
ax = axes[2]
ax.plot(iters, [v*100 for v in history["signal_efficiency"]], alpha=0.25, color="seagreen")
ax.plot(iters[sm_off:], smooth([v*100 for v in history["signal_efficiency"]]), color="seagreen", linewidth=2)
ax.axhline(baseline["mean_efficiency"]*100, color="red",   linestyle="--", linewidth=1.5, label=f"Before ({baseline['mean_efficiency']:.0%})")
ax.axhline(final["mean_efficiency"]*100,    color="green", linestyle="--", linewidth=1.5, label=f"After ({final['mean_efficiency']:.0%})")
ax.set_xlabel("Training Episode"); ax.set_ylabel("Signal Efficiency (%)")
ax.set_title("Signal Efficiency\n(only toggle wrong-phase signals)"); ax.set_ylim(0, 105); ax.legend(fontsize=8)

# Plot 4: Successful reroutes
ax = axes[3]
ax.plot(iters, history["mean_reroutes"], alpha=0.25, color="purple")
ax.plot(iters[sm_off:], smooth(history["mean_reroutes"]), color="purple", linewidth=2)
ax.axhline(baseline["mean_reroutes"], color="red",   linestyle="--", linewidth=1.5, label=f"Before ({baseline['mean_reroutes']:.1f})")
ax.axhline(final["mean_reroutes"],    color="green", linestyle="--", linewidth=1.5, label=f"After ({final['mean_reroutes']:.1f})")
ax.set_xlabel("Training Episode"); ax.set_ylabel("Reroutes per Episode")
ax.set_title("Adaptive Re-routing\n(agent avoids accidents & heavy traffic)"); ax.legend(fontsize=8)

plt.tight_layout()
out_path = Path("/content/ambulance_training_results.png")
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved → {out_path}")
print("Download from Colab Files sidebar (left panel) and commit to repo.")

In [ ]:
# CELL 11 — Cleanup
_server_proc.terminate()
print('Server stopped. Training complete.')